# TIPy Double Inverted Pendulum — PPO + MJX GPU

This notebook is the **native GPU version**.

The training path is intentionally different from the old Gym/Python-vector-env notebook:

```text
batched MJX physics
      ↓
batched observations
      ↓
PPO actor / critic
      ↓
batched actions
      ↓
batched MJX physics
```

The rollout is kept inside JAX with `vmap + jit + lax.scan`. There is no Python loop stepping one MuJoCo environment at a time and no GPU→CPU conversion on every physics step.

### Task

- Double inverted pendulum on a cart
- Cart usable range: **-2 m to +2 m**
- Continuous force: **-100 N to +100 N**
- Swing up both links, capture upright, balance
- PPO with a tanh-squashed Gaussian policy


## 1. Colab / GPU setup

MJX is a separate package from normal MuJoCo. The cell installs the JAX implementation, confirms CUDA, and mounts Drive for checkpoints.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# Recommended by the current MJX docs for NVIDIA GPU performance.
os.environ["XLA_FLAGS"] = (
    os.environ.get("XLA_FLAGS", "")
    + " --xla_gpu_triton_gemm_any=true"
)

if IN_COLAB or IN_KAGGLE:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--upgrade", "-q",
            "jax[cuda12]",
            "mujoco-mjx",
            "flax==0.12.8",
            "optax==0.2.8",
            "pandas>=2.0",
            "matplotlib>=3.8",
            "imageio>=2.34",
            "imageio-ffmpeg>=0.5",
        ],
        check=True,
    )

import jax
import jax.numpy as jnp
import numpy as np
import mujoco
from mujoco import mjx
import flax
import flax.linen as nn
from flax.training import train_state
import optax

print("JAX:", jax.__version__)
print("MuJoCo:", mujoco.__version__)
print("backend:", jax.default_backend())
print("devices:", jax.devices())

gpu_devices = [d for d in jax.devices() if d.platform == "gpu"]
if (IN_COLAB or IN_KAGGLE) and not gpu_devices:
    raise RuntimeError(
        "No GPU detected. In Colab choose Runtime → Change runtime type → GPU, "
        "restart the runtime, then run from the top."
    )


## 2. Training configuration

A tiny cart-pole scene does **not** benefit from running one MJX world at a time. The GPU version therefore starts with a large batch of parallel worlds.

`NUM_ENVS × ROLLOUT_STEPS` is the number of transitions in one PPO update.

The defaults below are deliberately conservative enough for a Colab GPU. If memory allows, `NUM_ENVS=4096` is a sensible later experiment.


In [ ]:
OUTPUT_DIR = (
    Path("/content/drive/MyDrive/ProjectsRuns/TIPy/runs/double/ppo-mjx/run-001")
    if IN_COLAB
    else Path.cwd() / "double-ppo-mjx-run"
)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
METRICS_PATH = OUTPUT_DIR / "metrics.csv"
DASHBOARD_PATH = OUTPUT_DIR / "dashboard.png"
REPLAY_PATH = OUTPUT_DIR / "replay.mp4"

SEED = 42

# Physics / task
MAX_EPISODE_STEPS = 5000
ACTION_LIMIT = 100.0
RAIL_LIMIT = 2.0

# GPU rollout
NUM_ENVS = 2048
ROLLOUT_STEPS = 64
TOTAL_STEPS = 20_000_000

# PPO
LEARNING_RATE = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPS = 0.2
VALUE_COEF = 0.5
ENTROPY_COEF = 0.005
MAX_GRAD_NORM = 0.5
PPO_EPOCHS = 6
NUM_MINIBATCHES = 16

# Logging
LOG_EVERY_UPDATES = 1
CHECKPOINT_EVERY_UPDATES = 10

BATCH_SIZE = NUM_ENVS * ROLLOUT_STEPS
if BATCH_SIZE % NUM_MINIBATCHES != 0:
    raise ValueError("NUM_ENVS * ROLLOUT_STEPS must divide evenly by NUM_MINIBATCHES.")

MINIBATCH_SIZE = BATCH_SIZE // NUM_MINIBATCHES

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("output:", OUTPUT_DIR)
print("parallel envs:", NUM_ENVS)
print("rollout steps/env:", ROLLOUT_STEPS)
print("transitions/update:", BATCH_SIZE)
print("minibatch size:", MINIBATCH_SIZE)


## 3. MuJoCo model

Two details matter for the GPU version:

1. All geoms have collisions disabled because this task does not need contact.
2. The cart joint itself is **not hard-limited**. Reaching `|x| >= 2 m` is handled as an RL terminal condition. This avoids paying for a joint-limit constraint in MJX.

The visual rail is still ±2 m and the observation exposes the cart position directly over `[-2, +2]`.


In [ ]:
MODEL_XML = r"""
<mujoco model="cartpole_double">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option
      timestep="0.005"
      gravity="0 0 -9.81"
      integrator="RK4"
      solver="Newton"
      iterations="1"
      ls_iterations="2"
      jacobian="dense">
    <flag eulerdamp="disable"/>
  </option>

  <default>
    <geom contype="0" conaffinity="0" />
  </default>

  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>

  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole1_mat" rgba="0.2 0.75 0.3 1" />
    <material name="pole2_mat" rgba="0.2 0.3 0.75 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>

  <worldbody>
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />

    <body name="frame">
      <geom name="tower_left" type="box" pos="-2.3 0 0.4" size="0.1 0.15 0.8" material="metal_mat" />
      <geom name="tower_right" type="box" pos="2.3 0 0.4" size="0.1 0.15 0.8" material="metal_mat" />
      <geom name="rail" type="box" pos="0 0 0.85" size="2 0.05 0.05" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
    </body>

    <body name="cart" pos="0 0 1">
      <joint
          name="cart_slide"
          type="slide"
          axis="1 0 0"
          limited="false"
          frictionloss="0"
          damping="0.1" />
      <inertial pos="0 0 0" mass="2" diaginertia="0.0333 0.0333 0.0333" />
      <geom name="cart_geom" type="box" size="0.085 0.08 0.1" mass="1.0" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.095 0"
            axisangle="1 0 0 1.5708" size="0.015 0.03" material="metal_mat" />

      <body name="pole1" pos="0 0.11 0" quat="6.12323399574e-17 0 -1 0">
        <joint name="pole1_hinge" type="hinge" axis="0 -1 0"
               frictionloss="0" damping="0.03"
               ref="3.1415926535897931" limited="false" />
        <inertial pos="0 0 0.15" mass="0.5"
                  diaginertia="0.0234541666667 0.0235041666667 8.33333333333e-05" />
        <site name="pole1_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole1_geom" type="box" pos="0 0 0.15"
              size="0.02 0.01 0.15" mass="0.5" material="pole1_mat" />
        <site name="pole1_tip_site" pos="0 0 0.3" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole2_mount_pin" type="capsule" pos="0 0.015 0.3"
              axisangle="1 0 0 1.5708" size="0.012 0.015" material="metal_mat" />

        <body name="pole2" pos="0 0.03 0.3" quat="1 0 0 0">
          <joint name="pole2_hinge" type="hinge" axis="0 -1 0"
                 frictionloss="0" damping="0.03" ref="0" limited="false" />
          <inertial pos="0 0 0.15" mass="0.5"
                    diaginertia="0.0234541666667 0.0235041666667 8.33333333333e-05" />
          <site name="pole2_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
          <geom name="pole2_geom" type="box" pos="0 0 0.15"
                size="0.02 0.01 0.15" mass="0.5" material="pole2_mat" />
          <site name="pole2_tip_site" pos="0 0 0.3" size="0.015" type="sphere" material="site_mat" />
        </body>
      </body>
    </body>

    <camera name="replay" pos="0 6 1.4" fovy="50"
            xyaxes="-1 0 0 0 -0.15 0.988686" />
  </worldbody>

  <actuator>
    <motor name="cart_motor" joint="cart_slide"
           gear="1" ctrlrange="-100 100" forcerange="-100 100" />
  </actuator>
</mujoco>
"""

mj_model = mujoco.MjModel.from_xml_string(MODEL_XML)

# Pure JAX MJX implementation. This is intentionally not the old CPU MjData.
mjx_model = mjx.put_model(mj_model, impl="jax")

print("nq:", mj_model.nq, "nv:", mj_model.nv, "nu:", mj_model.nu)
print("timestep:", mj_model.opt.timestep)


## 4. Pure-JAX batched environment

There is no `gym.Env` class here. Environment state is explicit JAX data.

Reset is vmapped across thousands of random keys. `step_batch` advances the entire batch in one compiled call.

### Observation

```text
0  x                         [-2, 2] m
1  clip(dx / 5)              [-1, 1]
2  cos(theta1)               [-1, 1]
3  sin(theta1)               [-1, 1]
4  cos(theta2)               [-1, 1]
5  sin(theta2)               [-1, 1]
6  clip(dtheta1 / 20)        [-1, 1]
7  clip(dtheta2 / 20)        [-1, 1]
```

The first entry is **not compressed to [-1,1]**. The network sees the actual cart coordinate from -2 m to +2 m.


In [ ]:
from flax import struct

OBS_DIM = 8
ACTION_DIM = 1

@struct.dataclass
class EnvState:
    data: object
    step_count: jax.Array
    episode_return: jax.Array
    episode_length: jax.Array
    captured_upright: jax.Array

def wrap_angle(x):
    return (x + jnp.pi) % (2.0 * jnp.pi) - jnp.pi

def physical_state(data):
    x = data.qpos[..., 0]
    relative1 = data.qpos[..., 1]
    relative2 = data.qpos[..., 2]

    dx = data.qvel[..., 0]
    relative_speed1 = data.qvel[..., 1]
    relative_speed2 = data.qvel[..., 2]

    theta1 = wrap_angle(relative1)
    theta2 = wrap_angle(relative1 + relative2)

    dtheta1 = relative_speed1
    dtheta2 = relative_speed1 + relative_speed2

    return x, theta1, theta2, dx, dtheta1, dtheta2

def observation_from_data(data):
    x, theta1, theta2, dx, dtheta1, dtheta2 = physical_state(data)

    return jnp.stack(
        [
            jnp.clip(x, -RAIL_LIMIT, RAIL_LIMIT),
            jnp.clip(dx / 5.0, -1.0, 1.0),
            jnp.cos(theta1),
            jnp.sin(theta1),
            jnp.cos(theta2),
            jnp.sin(theta2),
            jnp.clip(dtheta1 / 20.0, -1.0, 1.0),
            jnp.clip(dtheta2 / 20.0, -1.0, 1.0),
        ],
        axis=-1,
    ).astype(jnp.float32)

def _reset_one(key):
    key_pos, key_ang1, key_ang2, key_vel, key_w1, key_w2 = jax.random.split(key, 6)

    data = mjx.make_data(mjx_model)

    absolute1 = jnp.pi + jax.random.uniform(
        key_ang1, (), minval=-0.04, maxval=0.04
    )
    absolute2 = jnp.pi + jax.random.uniform(
        key_ang2, (), minval=-0.04, maxval=0.04
    )

    qpos = jnp.array(
        [
            jax.random.uniform(key_pos, (), minval=-0.01, maxval=0.01),
            absolute1,
            absolute2 - absolute1,
        ],
        dtype=jnp.float32,
    )

    absolute_speed1 = jax.random.uniform(
        key_w1, (), minval=-0.02, maxval=0.02
    )
    absolute_speed2 = jax.random.uniform(
        key_w2, (), minval=-0.02, maxval=0.02
    )

    qvel = jnp.array(
        [
            jax.random.uniform(key_vel, (), minval=-0.01, maxval=0.01),
            absolute_speed1,
            absolute_speed2 - absolute_speed1,
        ],
        dtype=jnp.float32,
    )

    data = data.replace(qpos=qpos, qvel=qvel)
    data = mjx.forward(mjx_model, data)
    return data

reset_data_batch = jax.jit(jax.vmap(_reset_one))

def reset_batch(keys):
    data = reset_data_batch(keys)
    n = keys.shape[0]
    state = EnvState(
        data=data,
        step_count=jnp.zeros((n,), dtype=jnp.int32),
        episode_return=jnp.zeros((n,), dtype=jnp.float32),
        episode_length=jnp.zeros((n,), dtype=jnp.int32),
        captured_upright=jnp.zeros((n,), dtype=jnp.bool_),
    )
    return state, observation_from_data(data)

def _step_one(data, action):
    # action is already tanh-squashed to [-1, 1].
    force = action[0] * ACTION_LIMIT
    ctrl = data.ctrl.at[0].set(force)
    data = data.replace(ctrl=ctrl)
    data = mjx.step(mjx_model, data)
    return data

step_data_batch = jax.vmap(_step_one)

def step_batch(state, action):
    next_data = step_data_batch(state.data, action)

    x, theta1, theta2, dx, dtheta1, dtheta2 = physical_state(next_data)
    next_obs_terminal = observation_from_data(next_data)

    normalized_force = action[..., 0]

    angle_reward = 0.5 * (jnp.cos(theta1) + jnp.cos(theta2))
    position_penalty = 0.25 * (x / RAIL_LIMIT) ** 2
    velocity_penalty = 0.01 * dx**2
    angular_velocity_penalty = 0.003 * (dtheta1**2 + dtheta2**2)
    control_penalty = 0.001 * normalized_force**2

    upright = (jnp.abs(theta1) < 0.35) & (jnp.abs(theta2) < 0.35)
    upright_bonus = jnp.where(upright, 3.0, 0.0)

    reward = (
        angle_reward
        - position_penalty
        - velocity_penalty
        - angular_velocity_penalty
        - control_penalty
        + upright_bonus
    )

    next_step_count = state.step_count + 1
    terminated = jnp.abs(x) >= RAIL_LIMIT
    truncated = next_step_count >= MAX_EPISODE_STEPS
    done = terminated | truncated

    reward = reward - jnp.where(terminated, 100.0, 0.0)

    episode_return = state.episode_return + reward
    episode_length = state.episode_length + 1
    captured = state.captured_upright | upright

    return (
        next_data,
        next_obs_terminal,
        reward.astype(jnp.float32),
        terminated,
        truncated,
        done,
        episode_return,
        episode_length,
        captured,
    )


## 5. PPO actor-critic

The policy is a Gaussian in an unconstrained latent action `u`, then:

\[
a = 	anh(u)
\]

so the physical command always lies in `[-1,+1]`.

That means:

- `a = -1` → `-100 N`
- `a = 0` → `0 N`
- `a = +1` → `+100 N`

Unlike clipping an ordinary Gaussian afterward, the PPO log-probability below includes the tanh Jacobian correction, so the action distribution used for learning matches the action actually applied to the cart.


In [ ]:
class ActorCritic(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, obs):
        x = nn.tanh(nn.Dense(256, kernel_init=nn.initializers.orthogonal(np.sqrt(2)))(obs))
        x = nn.tanh(nn.Dense(256, kernel_init=nn.initializers.orthogonal(np.sqrt(2)))(x))

        mean = nn.Dense(
            self.action_dim,
            kernel_init=nn.initializers.orthogonal(0.01),
            bias_init=nn.initializers.zeros_init(),
        )(x)

        log_std = self.param(
            "log_std",
            nn.initializers.constant(-0.5),
            (self.action_dim,),
        )
        log_std = jnp.clip(log_std, -5.0, 2.0)

        value = nn.Dense(
            1,
            kernel_init=nn.initializers.orthogonal(1.0),
            bias_init=nn.initializers.zeros_init(),
        )(x)

        return mean, log_std, jnp.squeeze(value, axis=-1)

network = ActorCritic(ACTION_DIM)

def gaussian_log_prob(raw_action, mean, log_std):
    std = jnp.exp(log_std)
    z = (raw_action - mean) / std
    return -0.5 * jnp.sum(
        z**2 + 2.0 * log_std + jnp.log(2.0 * jnp.pi),
        axis=-1,
    )

def squashed_log_prob(raw_action, action, mean, log_std):
    base_log_prob = gaussian_log_prob(raw_action, mean, log_std)
    correction = jnp.sum(
        jnp.log(jnp.clip(1.0 - action**2, 1e-6, 1.0)),
        axis=-1,
    )
    return base_log_prob - correction

def gaussian_entropy(log_std):
    return jnp.sum(log_std + 0.5 * jnp.log(2.0 * jnp.pi * jnp.e))

def sample_action(params, obs, key):
    mean, log_std, value = network.apply(params, obs)
    noise = jax.random.normal(key, mean.shape)
    raw_action = mean + jnp.exp(log_std) * noise
    action = jnp.tanh(raw_action)
    log_prob = squashed_log_prob(raw_action, action, mean, log_std)
    return raw_action, action, log_prob, value

def greedy_action(params, obs):
    mean, _, value = network.apply(params, obs)
    return jnp.tanh(mean), value


## 6. Train state and PPO loss


In [ ]:
class TrainState(train_state.TrainState):
    pass

rng = jax.random.PRNGKey(SEED)
rng, init_key = jax.random.split(rng)

params = network.init(
    init_key,
    jnp.zeros((1, OBS_DIM), dtype=jnp.float32),
)

optimizer = optax.chain(
    optax.clip_by_global_norm(MAX_GRAD_NORM),
    optax.adam(LEARNING_RATE),
)

state = TrainState.create(
    apply_fn=network.apply,
    params=params,
    tx=optimizer,
)

def _ppo_minibatch_update(state, batch):
    def loss_fn(params):
        mean, log_std, values = network.apply(params, batch["obs"])

        actions = jnp.tanh(batch["raw_action"])
        new_log_prob = squashed_log_prob(
            batch["raw_action"], actions, mean, log_std
        )

        log_ratio = new_log_prob - batch["old_log_prob"]
        ratio = jnp.exp(log_ratio)

        advantages = batch["advantage"]
        unclipped = ratio * advantages
        clipped = (
            jnp.clip(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS)
            * advantages
        )
        policy_loss = -jnp.mean(jnp.minimum(unclipped, clipped))

        value_pred_clipped = batch["old_value"] + jnp.clip(
            values - batch["old_value"],
            -CLIP_EPS,
            CLIP_EPS,
        )

        value_loss_unclipped = (values - batch["return"]) ** 2
        value_loss_clipped = (value_pred_clipped - batch["return"]) ** 2
        value_loss = 0.5 * jnp.mean(
            jnp.maximum(value_loss_unclipped, value_loss_clipped)
        )

        # Base Gaussian entropy. This is used as an exploration regularizer.
        entropy = gaussian_entropy(log_std)

        total_loss = (
            policy_loss
            + VALUE_COEF * value_loss
            - ENTROPY_COEF * entropy
        )

        approx_kl = jnp.mean(
            (jnp.exp(log_ratio) - 1.0) - log_ratio
        )
        clip_fraction = jnp.mean(
            (jnp.abs(ratio - 1.0) > CLIP_EPS).astype(jnp.float32)
        )

        aux = {
            "loss": total_loss,
            "policy_loss": policy_loss,
            "value_loss": value_loss,
            "entropy": entropy,
            "approx_kl": approx_kl,
            "clip_fraction": clip_fraction,
        }
        return total_loss, aux

    (_, aux), grads = jax.value_and_grad(
        loss_fn, has_aux=True
    )(state.params)

    state = state.apply_gradients(grads=grads)
    return state, aux

def _ppo_update_epochs(state, batch, rng):
    def epoch_step(carry, _):
        state, rng = carry
        rng, permutation_key = jax.random.split(rng)

        permutation = jax.random.permutation(
            permutation_key,
            BATCH_SIZE,
        )
        minibatch_indices = permutation.reshape(
            NUM_MINIBATCHES,
            MINIBATCH_SIZE,
        )

        def minibatch_step(state, indices):
            minibatch = jax.tree.map(
                lambda x: x[indices],
                batch,
            )
            state, metrics = _ppo_minibatch_update(
                state,
                minibatch,
            )
            return state, metrics

        state, metrics = jax.lax.scan(
            minibatch_step,
            state,
            minibatch_indices,
        )

        return (state, rng), metrics

    (state, rng), all_metrics = jax.lax.scan(
        epoch_step,
        (state, rng),
        xs=None,
        length=PPO_EPOCHS,
    )

    mean_metrics = jax.tree.map(
        jnp.mean,
        all_metrics,
    )
    return state, rng, mean_metrics

ppo_update_epochs = jax.jit(_ppo_update_epochs)


## 7. Fully compiled GPU rollout + GAE

This is the core architectural difference from the old notebook.

`collect_rollout` uses `lax.scan` for the time dimension, while each scan step advances all `NUM_ENVS` worlds together. No `np.asarray`, `float()`, `bool()`, or Python environment loop appears in the training hot path.

For GAE:

- rail hit (`terminated`) does **not** bootstrap
- time limit (`truncated`) **does** bootstrap from the terminal observation
- both terminate the temporal GAE recursion so two episodes are never stitched together


In [ ]:
@struct.dataclass
class Transition:
    obs: jax.Array
    raw_action: jax.Array
    action: jax.Array
    log_prob: jax.Array
    value: jax.Array
    reward: jax.Array
    terminated: jax.Array
    truncated: jax.Array
    done: jax.Array
    next_value: jax.Array
    completed_return: jax.Array
    completed_length: jax.Array
    completed_capture: jax.Array

def _collect_rollout(train_state, env_state, obs, rng):
    def one_step(carry, _):
        env_state, obs, rng = carry
        rng, action_key, reset_key = jax.random.split(rng, 3)

        raw_action, action, log_prob, value = sample_action(
            train_state.params, obs, action_key
        )

        (
            stepped_data,
            terminal_obs,
            reward,
            terminated,
            truncated,
            done,
            episode_return,
            episode_length,
            captured,
        ) = step_batch(env_state, action)

        # Value of the true post-step state BEFORE any reset.
        _, _, terminal_value = network.apply(
            train_state.params, terminal_obs
        )

        reset_keys = jax.random.split(reset_key, NUM_ENVS)
        reset_data = reset_data_batch(reset_keys)

        # Select reset state per environment.
        #
        # Data.where() with a vector mask does not broadcast correctly over
        # leaves such as qpos=(NUM_ENVS, 3).  Instead, vmap Data.where so each
        # environment receives a scalar boolean mask.
        next_data = jax.vmap(
            lambda stepped, reset, is_done: stepped.where(is_done, reset)
        )(stepped_data, reset_data, done)

        reset_obs = observation_from_data(reset_data)
        next_obs = jnp.where(done[:, None], reset_obs, terminal_obs)

        completed_return = jnp.where(done, episode_return, jnp.nan)
        completed_length = jnp.where(done, episode_length, 0)
        completed_capture = jnp.where(done, captured, False)

        next_env_state = EnvState(
            data=next_data,
            step_count=jnp.where(done, 0, env_state.step_count + 1),
            episode_return=jnp.where(done, 0.0, episode_return),
            episode_length=jnp.where(done, 0, episode_length),
            captured_upright=jnp.where(done, False, captured),
        )

        transition = Transition(
            obs=obs,
            raw_action=raw_action,
            action=action,
            log_prob=log_prob,
            value=value,
            reward=reward,
            terminated=terminated,
            truncated=truncated,
            done=done,
            next_value=terminal_value,
            completed_return=completed_return,
            completed_length=completed_length,
            completed_capture=completed_capture,
        )

        return (next_env_state, next_obs, rng), transition

    (env_state, obs, rng), traj = jax.lax.scan(
        one_step,
        (env_state, obs, rng),
        xs=None,
        length=ROLLOUT_STEPS,
    )

    return env_state, obs, rng, traj

collect_rollout = jax.jit(_collect_rollout)

@jax.jit
def compute_gae(traj):
    def reverse_step(gae, t):
        # Bootstrap at ordinary transitions and at time-limit truncations,
        # but never across a true terminal rail failure.
        bootstrap_mask = 1.0 - t.terminated.astype(jnp.float32)
        recursion_mask = 1.0 - t.done.astype(jnp.float32)

        delta = (
            t.reward
            + GAMMA * bootstrap_mask * t.next_value
            - t.value
        )

        gae = (
            delta
            + GAMMA * GAE_LAMBDA * recursion_mask * gae
        )
        return gae, gae

    _, advantages = jax.lax.scan(
        reverse_step,
        jnp.zeros((NUM_ENVS,), dtype=jnp.float32),
        traj,
        reverse=True,
    )

    returns = advantages + traj.value

    advantages = (
        advantages - advantages.mean()
    ) / (advantages.std() + 1e-8)

    return advantages, returns

def flatten_rollout(traj, advantages, returns):
    def flat(x):
        return x.reshape((BATCH_SIZE,) + x.shape[2:])

    return {
        "obs": flat(traj.obs),
        "raw_action": flat(traj.raw_action),
        "old_log_prob": flat(traj.log_prob),
        "old_value": flat(traj.value),
        "advantage": flat(advantages),
        "return": flat(returns),
    }


## 8. Checkpoints and metrics

Only update-level summaries come back to Python. Physics transitions themselves stay on the accelerator.


In [ ]:
import csv
import pickle
import time
from datetime import datetime, timezone

METRIC_FIELDS = [
    "timestamp",
    "update",
    "total_steps",
    "reward",
    "policy_loss",
    "value_loss",
    "entropy",
    "approx_kl",
    "clip_fraction",
    "episode_length",
    "steps_per_second",
    "both_upright",
    "episodes_finished",
]

def save_checkpoint(train_state, rng, update, total_steps):
    payload = {
        "version": 2,
        "update": int(update),
        "total_steps": int(total_steps),
        "params": jax.device_get(train_state.params),
        "opt_state": jax.device_get(train_state.opt_state),
        "rng": np.asarray(jax.device_get(rng)),
    }

    final_path = CHECKPOINT_DIR / f"checkpoint_{total_steps:012d}.pkl"
    tmp_path = final_path.with_suffix(".tmp")

    with open(tmp_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, final_path)

    latest = CHECKPOINT_DIR / "latest.pkl"
    latest_tmp = CHECKPOINT_DIR / "latest.tmp"
    with open(latest_tmp, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.flush()
        os.fsync(f.fileno())
    os.replace(latest_tmp, latest)

    return final_path

def restore_checkpoint(train_state):
    latest = CHECKPOINT_DIR / "latest.pkl"
    if not latest.exists():
        return train_state, jax.random.PRNGKey(SEED), 0, 0

    with open(latest, "rb") as f:
        payload = pickle.load(f)

    train_state = train_state.replace(
        params=jax.device_put(payload["params"]),
        opt_state=jax.device_put(payload["opt_state"]),
    )

    restored_rng = jnp.asarray(payload["rng"], dtype=jnp.uint32)
    print(
        "Restored:",
        "update", payload["update"],
        "| steps", payload["total_steps"],
    )
    return (
        train_state,
        restored_rng,
        int(payload["update"]),
        int(payload["total_steps"]),
    )

def append_metric(row):
    exists = METRICS_PATH.exists()
    with open(METRICS_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=METRIC_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


## 9. GPU smoke test and benchmark

Run this **before training**.

The first call includes XLA compilation and can be slow. The second call measures the compiled rollout.

For this notebook, the meaningful benchmark is a **batched rollout**, not one MJX world stepped from Python.


In [ ]:
state, rng, update_index, total_steps = restore_checkpoint(state)

rng, env_key, rollout_key = jax.random.split(rng, 3)
env_keys = jax.random.split(env_key, NUM_ENVS)
env_state, obs = reset_batch(env_keys)

print("Compiling first GPU rollout...")
t0 = time.perf_counter()
test_env_state, test_obs, test_rng, test_traj = collect_rollout(
    state, env_state, obs, rollout_key
)
jax.block_until_ready(test_traj.reward)
compile_and_run = time.perf_counter() - t0
print(f"first rollout (compile + run): {compile_and_run:.2f} s")

print("Timing compiled GPU rollout...")
t0 = time.perf_counter()
test_env_state, test_obs, test_rng, test_traj = collect_rollout(
    state, test_env_state, test_obs, test_rng
)
jax.block_until_ready(test_traj.reward)
elapsed = time.perf_counter() - t0

transitions = NUM_ENVS * ROLLOUT_STEPS
print(f"compiled rollout: {elapsed:.4f} s")
print(f"physics/policy throughput: {transitions / elapsed:,.0f} transitions/s")
print("mean reward:", float(jnp.mean(test_traj.reward)))
print("device:", test_traj.reward.device)


## 10. PPO training

Each iteration:

1. one compiled GPU rollout
2. GPU GAE
3. flatten the rollout without leaving device memory
4. shuffled PPO minibatches
5. copy only scalar metrics back to the host

If the runtime disconnects, rerunning from the top restores `latest.pkl`.


In [ ]:
# Fresh environment state for training. The policy/optimizer may already be restored.
rng, env_key = jax.random.split(rng)
env_state, obs = reset_batch(jax.random.split(env_key, NUM_ENVS))

session_start = time.perf_counter()
session_start_steps = total_steps

print(
    f"Training from update={update_index}, steps={total_steps:,} "
    f"toward {TOTAL_STEPS:,}"
)

try:
    while total_steps < TOTAL_STEPS:
        update_index += 1

        # 1) Fully compiled GPU rollout.
        env_state, obs, rng, traj = collect_rollout(
            state, env_state, obs, rng
        )

        # 2) GPU GAE and flattening.
        advantages, returns = compute_gae(traj)
        batch = flatten_rollout(traj, advantages, returns)

        # 3) All PPO epochs + minibatches are also executed through lax.scan
        #    inside one JIT-compiled update function.
        state, rng, metric_mean = ppo_update_epochs(
            state,
            batch,
            rng,
        )

        # Synchronize once per complete PPO update.
        jax.block_until_ready(state.params)

        total_steps += BATCH_SIZE

        # Only completed-episode summaries cross back to the host.
        completed_returns = np.asarray(
            jax.device_get(traj.completed_return)
        ).reshape(-1)
        completed_lengths = np.asarray(
            jax.device_get(traj.completed_length)
        ).reshape(-1)
        completed_captures = np.asarray(
            jax.device_get(traj.completed_capture)
        ).reshape(-1)

        valid = np.isfinite(completed_returns)
        episodes_finished = int(valid.sum())

        if episodes_finished:
            mean_ep_return = float(completed_returns[valid].mean())
            mean_ep_length = float(completed_lengths[valid].mean())
            capture_rate = float(completed_captures[valid].mean())
        else:
            mean_ep_return = float("nan")
            mean_ep_length = float("nan")
            capture_rate = float("nan")

        metric_mean = jax.device_get(metric_mean)

        elapsed = time.perf_counter() - session_start
        sps = (
            (total_steps - session_start_steps) / elapsed
            if elapsed > 0 else 0.0
        )

        row = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "update": update_index,
            "total_steps": total_steps,
            "reward": mean_ep_return,
            "policy_loss": float(metric_mean["policy_loss"]),
            "value_loss": float(metric_mean["value_loss"]),
            "entropy": float(metric_mean["entropy"]),
            "approx_kl": float(metric_mean["approx_kl"]),
            "clip_fraction": float(metric_mean["clip_fraction"]),
            "episode_length": mean_ep_length,
            "steps_per_second": sps,
            "both_upright": capture_rate,
            "episodes_finished": episodes_finished,
        }
        append_metric(row)

        if update_index % LOG_EVERY_UPDATES == 0:
            print(
                f"u={update_index:05d} "
                f"steps={total_steps:>10,} "
                f"epR={mean_ep_return:>9.2f} "
                f"len={mean_ep_length:>7.1f} "
                f"capture={capture_rate:>6.1%} "
                f"KL={float(metric_mean['approx_kl']):.4f} "
                f"SPS={sps:,.0f}"
            )

        if update_index % CHECKPOINT_EVERY_UPDATES == 0:
            path_ckpt = save_checkpoint(
                state, rng, update_index, total_steps
            )
            print("checkpoint:", path_ckpt)

except KeyboardInterrupt:
    path_ckpt = save_checkpoint(
        state, rng, update_index, total_steps
    )
    print("\nInterrupted safely.")
    print("checkpoint:", path_ckpt)
    raise


## 11. Training dashboard


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if not METRICS_PATH.exists():
    raise FileNotFoundError("No metrics.csv yet. Run training first.")

metrics = (
    pd.read_csv(METRICS_PATH)
    .drop_duplicates("update", keep="last")
    .sort_values("update")
)

window = min(20, len(metrics))
x = metrics["update"]

figure, axes = plt.subplots(
    2, 3,
    figsize=(16, 9),
    constrained_layout=True,
)
figure.suptitle(
    "TIPy Double Pendulum — PPO + MJX GPU",
    fontsize=16,
    fontweight="bold",
)

def smooth(series):
    return series.rolling(window, min_periods=1).mean()

axes[0, 0].plot(x, metrics["reward"], alpha=0.25)
axes[0, 0].plot(x, smooth(metrics["reward"]))
axes[0, 0].set_title("Episode return")

axes[0, 1].plot(x, metrics["both_upright"])
axes[0, 1].set_title("Both-upright capture rate")
axes[0, 1].set_ylim(0, 1)

axes[0, 2].plot(x, metrics["episode_length"])
axes[0, 2].set_title("Episode length")

axes[1, 0].plot(x, metrics["policy_loss"], label="policy")
axes[1, 0].plot(x, metrics["value_loss"], label="value")
axes[1, 0].legend()
axes[1, 0].set_title("PPO losses")

axes[1, 1].plot(x, metrics["entropy"], label="entropy")
axes[1, 1].plot(x, metrics["approx_kl"], label="approx KL")
axes[1, 1].legend()
axes[1, 1].set_title("Policy diagnostics")

axes[1, 2].plot(x, metrics["steps_per_second"])
axes[1, 2].set_title("End-to-end transitions / second")

for ax in axes.flat:
    ax.grid(alpha=0.2)
    ax.set_xlabel("update")

figure.savefig(DASHBOARD_PATH, dpi=160)
plt.show()

print("saved:", DASHBOARD_PATH)


## 12. Batched deterministic GPU evaluation

Evaluation also stays batched on MJX. It runs many deterministic episodes in parallel using the policy mean.

This avoids the single-world MJX performance trap.


In [ ]:
EVAL_ENVS = 256
EVAL_MAX_STEPS = MAX_EPISODE_STEPS

@jax.jit
def run_eval(params, keys):
    data = reset_data_batch(keys)
    obs = observation_from_data(data)

    returns = jnp.zeros((EVAL_ENVS,), dtype=jnp.float32)
    lengths = jnp.zeros((EVAL_ENVS,), dtype=jnp.int32)
    done = jnp.zeros((EVAL_ENVS,), dtype=jnp.bool_)
    captured = jnp.zeros((EVAL_ENVS,), dtype=jnp.bool_)

    def eval_step(carry, _):
        data, obs, returns, lengths, done, captured = carry

        action, _ = greedy_action(params, obs)
        next_data = step_data_batch(data, action)

        x, theta1, theta2, dx, dtheta1, dtheta2 = physical_state(next_data)

        angle_reward = 0.5 * (jnp.cos(theta1) + jnp.cos(theta2))
        reward = (
            angle_reward
            - 0.25 * (x / RAIL_LIMIT) ** 2
            - 0.01 * dx**2
            - 0.003 * (dtheta1**2 + dtheta2**2)
            - 0.001 * action[..., 0]**2
            + jnp.where(
                (jnp.abs(theta1) < 0.35)
                & (jnp.abs(theta2) < 0.35),
                3.0,
                0.0,
            )
        )

        terminated = jnp.abs(x) >= RAIL_LIMIT
        reward = reward - jnp.where(terminated, 100.0, 0.0)

        active = ~done
        returns = returns + jnp.where(active, reward, 0.0)
        lengths = lengths + active.astype(jnp.int32)

        upright = (
            (jnp.abs(theta1) < 0.35)
            & (jnp.abs(theta2) < 0.35)
        )
        captured = captured | (active & upright)

        new_done = done | terminated | (lengths >= EVAL_MAX_STEPS)

        # Finished worlds remain numerically stepped, but their metrics are frozen.
        next_obs = observation_from_data(next_data)

        return (
            next_data,
            next_obs,
            returns,
            lengths,
            new_done,
            captured,
        ), None

    final, _ = jax.lax.scan(
        eval_step,
        (data, obs, returns, lengths, done, captured),
        xs=None,
        length=EVAL_MAX_STEPS,
    )

    _, _, returns, lengths, done, captured = final
    return returns, lengths, captured

rng, eval_key = jax.random.split(rng)
eval_keys = jax.random.split(eval_key, EVAL_ENVS)

print("Running batched GPU evaluation...")
eval_returns, eval_lengths, eval_captures = run_eval(
    state.params,
    eval_keys,
)
jax.block_until_ready(eval_returns)

print("mean return:", float(eval_returns.mean()))
print("mean length:", float(eval_lengths.mean()))
print("capture rate:", float(eval_captures.mean()))


## 13. Optional replay video

Training and evaluation above are MJX GPU.

For an MP4, this final cell intentionally uses ordinary MuJoCo only as a **renderer / one-episode visualizer**. It does not affect training throughput or learned parameters. A single MJX world is precisely the workload MJX-JAX is poor at, so there is no reason to force the visualization path through it.


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

REPLAY_SEED = 50_004
REPLAY_SECONDS = 10.0
REPLAY_FPS = 50

# CPU MuJoCo is used only to produce pixels for one visual replay.
replay_model = mujoco.MjModel.from_xml_string(MODEL_XML)
replay_data = mujoco.MjData(replay_model)

rng_np = np.random.default_rng(REPLAY_SEED)
absolute1 = np.pi + rng_np.uniform(-0.04, 0.04)
absolute2 = np.pi + rng_np.uniform(-0.04, 0.04)

replay_data.qpos[:] = [
    rng_np.uniform(-0.01, 0.01),
    absolute1,
    absolute2 - absolute1,
]

absolute_speed1 = rng_np.uniform(-0.02, 0.02)
absolute_speed2 = rng_np.uniform(-0.02, 0.02)

replay_data.qvel[:] = [
    rng_np.uniform(-0.01, 0.01),
    absolute_speed1,
    absolute_speed2 - absolute_speed1,
]

mujoco.mj_forward(replay_model, replay_data)

renderer = mujoco.Renderer(
    replay_model,
    height=480,
    width=640,
)

writer = imageio.get_writer(
    REPLAY_PATH,
    fps=REPLAY_FPS,
    codec="libx264",
    quality=8,
)

frame_stride = max(
    1,
    round(1.0 / (replay_model.opt.timestep * REPLAY_FPS)),
)

def cpu_obs(data):
    x = float(data.qpos[0])
    relative1 = float(data.qpos[1])
    relative2 = float(data.qpos[2])
    dx = float(data.qvel[0])
    w1 = float(data.qvel[1])
    w2rel = float(data.qvel[2])

    theta1 = (relative1 + np.pi) % (2 * np.pi) - np.pi
    theta2 = (relative1 + relative2 + np.pi) % (2 * np.pi) - np.pi

    return np.array(
        [
            np.clip(x, -RAIL_LIMIT, RAIL_LIMIT),
            np.clip(dx / 5.0, -1.0, 1.0),
            np.cos(theta1),
            np.sin(theta1),
            np.cos(theta2),
            np.sin(theta2),
            np.clip(w1 / 20.0, -1.0, 1.0),
            np.clip((w1 + w2rel) / 20.0, -1.0, 1.0),
        ],
        dtype=np.float32,
    )

num_steps = int(REPLAY_SECONDS / replay_model.opt.timestep)

for step in range(num_steps):
    obs_cpu = cpu_obs(replay_data)
    action, _ = greedy_action(
        state.params,
        jnp.asarray(obs_cpu)[None, :],
    )
    force = float(np.asarray(jax.device_get(action))[0, 0]) * ACTION_LIMIT

    replay_data.ctrl[0] = force
    mujoco.mj_step(replay_model, replay_data)

    if step % frame_stride == 0:
        renderer.update_scene(replay_data, camera="replay")
        writer.append_data(renderer.render())

    if abs(float(replay_data.qpos[0])) >= RAIL_LIMIT:
        break

writer.close()
renderer.close()

print("saved:", REPLAY_PATH)
display(Video(str(REPLAY_PATH), embed=True))
